In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
csv_path = os.path.join(path, "Q1_data.csv")
df = pd.read_csv(csv_path)


In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
import matplotlib.pyplot as plt

df["Delivery_Time"].hist(bins=30, edgecolor='black')
plt.title(f"Target Distribution (delivery_time)")
plt.xlabel("delivery_time")
plt.ylabel("Frequency")
plt.grid(False)

plt.show()

In [ ]:
df = df.drop("Order_ID", axis=1).copy()

In [ ]:

df["Delivery_Time"] = df["Delivery_Time"].fillna(df["Delivery_Time"].mean())
df["Weather"] = df["Weather"].fillna("Unknown")
df["Traffic_Level"] = df["Traffic_Level"].fillna("Unknown")
df["Time_of_Day"] = df["Time_of_Day"].fillna("Unknown")
df["Courier_Experience_yrs"] = df["Courier_Experience_yrs"].fillna(df["Courier_Experience_yrs"].mean())
df.isna().sum()

In [ ]:
print(df.duplicated().sum())

df = df.drop_duplicates()
print(df.duplicated().sum())

In [ ]:
from sklearn.preprocessing import LabelEncoder
df.info()
categorical_cols = df.select_dtypes(include=["object"]).columns

for col in categorical_cols:
    encoder = LabelEncoder()
    df[col] = encoder.fit_transform(df[col])
df.info()

In [ ]:
from sklearn.preprocessing import StandardScaler
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()

In [ ]:
df["Delivery_Time"].value_counts()

#The target is imbalanced

In [ ]:
X = df.drop("Delivery_Time",axis=1).copy()
y = df["Delivery_Time"].copy()

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

lr_mae =[]
maes = 0
model = RandomForestRegressor(n_estimators=100,max_depth=20, random_state=42)
n_splits = 10
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
y_pred =0
for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    # Train
  model.fit(X_train,y_train)

  # Predict
  y_pred = model.predict(X_test)
  # Calculate evaluation metrics
  mae = mean_absolute_error(y_test, y_pred)

  maes = maes+mae
  # Store results
  lr_mae.append(mae)

print(f"Avg: {maes/n_splits}")

In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': numerical_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
plt.hist(y_pred)
plt.show()

In [ ]:
# Task Bonus: Write your code here: